# 04 — Delta Lake production patterns and Iceberg comparison (EMR)

Demonstrate `DESCRIBE HISTORY`, `MERGE`, time travel, `OPTIMIZE`/`ZORDER`, `VACUUM`, and a managed Iceberg table for comparison, against the tables built in `01`-`03`.

Run the cell below first, before anything else -- it configures Delta Lake for this notebook's Spark session (EMR doesn't bundle Delta by default, unlike Databricks). `%%configure -f` must run before any other Spark code in this session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "retail_lakehouse"
base_path = "s3://<your-lakehouse-bucket>/data"

from retail_lakehouse.config import PipelineConfig
cfg = PipelineConfig(schema=schema, base_path=base_path)
spark.sql(f"USE `{cfg.schema}`")

## Table history

In [ ]:
spark.sql(f"DESCRIBE HISTORY {cfg.table('silver_clickstream_batch')}").toPandas()

## MERGE upsert into a customer dimension (CDC-style update)

In [ ]:
spark.table(cfg.table("dim_customer_seed")).write.mode("overwrite").format("delta").saveAsTable(cfg.table("dim_customer_current"))

updates = spark.createDataFrame([
    ("u00001", "loyal", "west", 1704067200),
    ("u99999", "new", "central", 1735689600),
], "user_id string, segment string, region string, signup_epoch long")
updates.createOrReplaceTempView("customer_updates")

spark.sql(f"""
MERGE INTO {cfg.table("dim_customer_current")} AS t
USING customer_updates AS s
ON t.user_id = s.user_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

spark.table(cfg.table("dim_customer_current")).filter("user_id in ('u00001','u99999')").toPandas()

## Time travel

In [ ]:
history = spark.sql(f"DESCRIBE HISTORY {cfg.table('dim_customer_current')}")
versions = [r.version for r in history.select("version").collect()]
if len(versions) >= 2:
    earliest = min(versions)
    print("Reading version", earliest, "(before the MERGE):")
    print(spark.read.option("versionAsOf", earliest).table(cfg.table("dim_customer_current")).limit(10).toPandas())

## OPTIMIZE + ZORDER and VACUUM

`OPTIMIZE`/`ZORDER` require a delta-spark version that supports them (2.0+ for OPTIMIZE, later for ZORDER) -- this cell degrades gracefully if not. `VACUUM` physically deletes files no longer referenced by the log and older than the retention threshold (default 7 days) -- run it on a schedule, not ad hoc, and never with a retention below 7 days on a table with concurrent readers/time-travel users.

In [ ]:
try:
    spark.sql(f"OPTIMIZE {cfg.table('silver_clickstream_batch')} ZORDER BY (user_id, product_id)")
except Exception as e:
    print("OPTIMIZE/ZORDER not supported by this delta-spark version:", str(e)[:300])

spark.sql(f"DESCRIBE DETAIL {cfg.table('silver_clickstream_batch')}").toPandas()

## Managed Iceberg table for comparison

Falls back to a Delta comparison table if this EMR cluster's Spark session isn't configured with an Iceberg catalog (this project's EMR clusters configure Glue as a Hive-compatible metastore for Delta, not an Iceberg catalog -- so this fallback path is expected to run by default here, exactly as it would on a Databricks workspace without managed Iceberg enabled).

In [ ]:
iceberg_table = cfg.table("iceberg_clickstream_sample")
fallback_delta = cfg.table("delta_iceberg_comparison_sample")
source = spark.table(cfg.table("silver_clickstream_batch")).limit(200)

try:
    spark.sql(f"DROP TABLE IF EXISTS {iceberg_table}")
    spark.sql(f"""
    CREATE TABLE {iceberg_table} (
      event_id STRING, event_ts TIMESTAMP, user_id STRING, product_id STRING,
      event_type STRING, event_date DATE
    ) USING ICEBERG
    """)
    (source.select("event_id", "event_ts", "user_id", "product_id", "event_type", "event_date")
          .write.mode("append").saveAsTable(iceberg_table))
    print("Created managed Iceberg table:", iceberg_table)
except Exception as e:
    print("Managed Iceberg not available on this cluster, falling back to Delta:", str(e)[:300])
    (source.select("event_id", "event_ts", "user_id", "product_id", "event_type", "event_date")
          .write.mode("overwrite").format("delta").saveAsTable(fallback_delta))

## Next

`05_observability_testing_performance.ipynb` covers monitoring these tables and streaming queries in production, and `06_capstone_end_to_end.ipynb` ties the whole pipeline together in one flow.